In [54]:
!pip install -U langgraph langchain langchain-google-genai langchain-community langchain-chroma pypdf

Defaulting to user installation because normal site-packages is not writeable


In [55]:
from typing import TypedDict
import os
from dotenv import load_dotenv


from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings

from langchain_chroma import Chroma

from langgraph.graph import StateGraph, START, END

# Explanation : https://chatgpt.com/share/6a9fe027-0e58-83ee-8e2c-36c845504fe6

In [56]:
load_dotenv()

True

In [57]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

In [58]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# text - embeding - vector - store 

In [59]:
loader = PyPDFLoader("python.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 4


In [60]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [61]:
chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 16


In [62]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="chroma_db"
)

# mongodb = table  //collection

In [63]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [64]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="chroma_db",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

vectorstore.add_documents(chunks)

print("Chroma DB created successfully!")

Chroma DB created successfully!


In [65]:
class State(TypedDict):
    question: str
    documents: list
    answer: str

In [66]:
def retrieve_node(state: State):

    question = state["question"]

    documents = retriever.invoke(question)

    return {
        "documents": documents
    }

In [67]:
def generate_node(state: State):

    question = state["question"]

    documents = state["documents"]

    context = "\n\n".join(
        doc.page_content
        for doc in documents
    )
#temlate ""ai - instruction 
    prompt = f"""
    Answer the question using only the context below.

    Context:
    {context}

    Question:
    {question}

    If the answer is not present in the context,
    say "I don't know based on the provided document."
    """

    response = llm.invoke(prompt)

    return {
        "answer": response.content[0]["text"]
    }


In [68]:
graph = StateGraph(State)

In [69]:
graph.add_node("retrieve", retrieve_node)

graph.add_node("generate", generate_node)

In [70]:
graph.add_edge(START, "retrieve")

graph.add_edge("retrieve", "generate")

graph.add_edge("generate", END)

In [71]:
app = graph.compile()

In [72]:
result = app.invoke({
    "question": "What is java",
    "documents": [],
    "answer": ""
})

In [73]:
print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

Question:
What is java

Answer:
I don't know based on the provided document.
